<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/EXIF_Metadata_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python utility for forensic examination of photographs by extracting available EXIF metadata, categorizing the information into camera details, image properties, timestamps, and GPS information, while handling missing metadata fields without terminating the program.

**Algorithm**

Obtain an image file.

Open the image using the Python Imaging Library (PIL).

Read the available EXIF metadata.
Decode EXIF tag numbers into meaningful names.

Categorize metadata into camera, image, timestamp, and GPS information.

Check whether GPS coordinates are available.

Convert GPS coordinates into readable latitude and longitude values.

Display GPS information separately because it may have forensic significance.

Handle missing or unavailable metadata fields safely.

Display the complete categorized forensic report.

In [6]:
# ==============================================
# FORENSIC EXIF METADATA ANALYZER
# ==============================================

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

# ----------------------------------------------
# 1. Upload Image in Google Colab
# ----------------------------------------------

from google.colab import files

uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Please upload an image file to proceed.")
    raise SystemExit

image_file = list(uploaded.keys())[0]

# ----------------------------------------------
# 2. Open Image
# ----------------------------------------------

try:
    image = Image.open(image_file)
    print("=" * 70)
    print("          FORENSIC IMAGE / EXIF ANALYSIS")
    print("=" * 70)

    print(f"\nFile Name : {image_file}")
    print(f"Image Size: {image.size}")
    print(f"Image Mode: {image.mode}")

except Exception as e:
    print("Error opening image:", e)
    raise SystemExit

# ----------------------------------------------
# 3. Extract EXIF Metadata
# ----------------------------------------------

exif_data = image.getexif()

if not exif_data:
    print("\nNo EXIF metadata available.")
    # The user requested to continue execution even if no EXIF data is found.
    # So, we don't raise SystemExit here.
    metadata = {}
else:
    metadata = {}
    for tag_id, value in exif_data.items():
        tag_name = TAGS.get(tag_id, str(tag_id))
        metadata[tag_name] = value

# ----------------------------------------------
# 4. Categories
# ----------------------------------------------

camera_tags = [
    "Make",
    "Model",
    "LensModel",
    "LensMake",
    "Software",
    "Artist"
]

image_tags = [
    "ImageWidth",
    "ImageLength",
    "Orientation",
    "XResolution",
    "YResolution",
    "ResolutionUnit",
    "ColorSpace",
    "Flash",
    "FocalLength"
]

time_tags = [
    "DateTime",
    "DateTimeOriginal",
    "DateTimeDigitized"
]

# ----------------------------------------------
# 5. Display Camera Information
# ----------------------------------------------

print("\n" + "-" * 70)
print("CAMERA DETAILS")
print("-" * 70)

found = False

for tag in camera_tags:
    if tag in metadata:
        print(f"{tag:20}: {metadata[tag]}")
        found = True

if not found:
    print("No camera details available.")

# ----------------------------------------------
# 6. Display Image Properties
# ----------------------------------------------

print("\n" + "-" * 70)
print("IMAGE PROPERTIES")
print("-" * 70)

found = False

for tag in image_tags:
    if tag in metadata:
        print(f"{tag:20}: {metadata[tag]}")
        found = True

if not found:
    print("No additional image properties available.")

# ----------------------------------------------
# 7. Display Timestamps
# ----------------------------------------------

print("\n" + "-" * 70)
print("TIMESTAMPS")
print("-" * 70)

found = False

for tag in time_tags:
    if tag in metadata:
        print(f"{tag:20}: {metadata[tag]}")
        found = True

if not found:
    print("No timestamp metadata available.")

# ----------------------------------------------
# 8. Extract GPS Information
# ----------------------------------------------

gps_info = exif_data.get(34853) if exif_data else None

print("\n" + "-" * 70)
print("GPS / LOCATION INFORMATION")
print("-" * 70)

if gps_info:

    gps_data = {}

    for key, value in gps_info.items():
        gps_data[GPSTAGS.get(key, key)] = value

    for key, value in gps_data.items():
        print(f"{key:20}: {value}")

    # ------------------------------------------
    # Convert GPS Coordinates
    # ------------------------------------------

    def convert_to_degrees(value):

        d = float(value[0])
        m = float(value[1])
        s = float(value[2])

        return d + (m / 60.0) + (s / 3600.0)

    try:
        lat = convert_to_degrees(
            gps_data["GPSLatitude"]
        )

        lon = convert_to_degrees(
            gps_data["GPSLongitude"]
        )

        if gps_data.get("GPSLatitudeRef") == "S":
            lat = -lat

        if gps_data.get("GPSLongitudeRef") == "W":
            lon = -lon

        print("\nGPS COORDINATES")
        print(f"Latitude : {lat:.6f}")
        print(f"Longitude: {lon:.6f}")

        print("\nFORENSIC SIGNIFICANCE:")
        print("GPS coordinates may provide information about")
        print("the location where the photograph was captured.")

    except Exception:
        print("\nGPS data exists, but coordinates could not")
        print("be converted into decimal format.")

else:
    print("No GPS information available.")

# ----------------------------------------------
# 9. Display Other EXIF Metadata
# ----------------------------------------------

print("\n" + "-" * 70)
print("OTHER AVAILABLE EXIF INFORMATION")
print("-" * 70)

known_tags = camera_tags + image_tags + time_tags

other_found = False

if exif_data: # Only iterate if exif_data actually exists
    for tag, value in metadata.items():

        if tag not in known_tags and tag != 'GPSInfo': # Also exclude 'GPSInfo' from other tags
            print(f"{tag:25}: {value}")
            other_found = True

if not other_found:
    print("No additional EXIF fields available.")

print("\n" + "=" * 70)
print("EXIF FORENSIC ANALYSIS COMPLETED")
print("=" * 70)

Saving 192465014 - Vanashri.S.R Part 1.png to 192465014 - Vanashri.S.R Part 1 (1).png
          FORENSIC IMAGE / EXIF ANALYSIS

File Name : 192465014 - Vanashri.S.R Part 1 (1).png
Image Size: (1920, 1080)
Image Mode: RGBA

No EXIF metadata available.

----------------------------------------------------------------------
CAMERA DETAILS
----------------------------------------------------------------------
No camera details available.

----------------------------------------------------------------------
IMAGE PROPERTIES
----------------------------------------------------------------------
No additional image properties available.

----------------------------------------------------------------------
TIMESTAMPS
----------------------------------------------------------------------
No timestamp metadata available.

----------------------------------------------------------------------
GPS / LOCATION INFORMATION
----------------------------------------------------------------------
No 

**Result**

The Python utility successfully extracted and categorized the available EXIF metadata from the photograph into camera details, image properties, timestamps, and location information. When GPS metadata was present, the program separately displayed the latitude and longitude coordinates and highlighted their potential forensic significance. The utility also handled missing EXIF fields without terminating, making it suitable for basic forensic examination of photographs.